In [ ]:
import numpy as np

#------------------------------------HSIC--------------------------------------

def com_HSIC(data_X,data_Y,n,d1,d2):
    distance_X=(data_X@data_X.T)**d1
    distance_Y=(data_Y@data_Y.T)**d2
    atau=np.sum(np.diag(distance_X@distance_Y))\
    +np.ones(distance_X.shape[0])@distance_X@np.ones(distance_X.shape)@distance_Y@np.ones(distance_Y.shape[0])/(n-1)/(n-2)\
    -2*np.ones(distance_X.shape[0])@distance_X@distance_Y@np.ones(distance_X.shape[0])/(n-2)
    
    atau=atau/n/(n-3)
    
    datay_copy=np.copy(data_Y)
   
    p1_value=[]
    for i in range(100):
        np.random.shuffle(datay_copy)
        distance_Y=datay_copy@datay_copy.T
        phsic_value=np.sum(np.diag((distance_X@distance_Y)))\
        +np.ones(distance_X.shape[0])@distance_X@np.ones(distance_X.shape)@distance_Y@np.ones(distance_Y.shape[0])/(n-1)/(n-2)\
        -2*np.ones(distance_X.shape[0])@distance_X@distance_Y@np.ones(distance_X.shape[0])/(n-2)

        p1_value.append(phsic_value/n/(n-3))
            
            
    return atau, p1_value



import rpy2.robjects as robjects
robjects.r("library(dcov)")



def com_PCOV(data_X,data_Y,n):
    a=robjects.FloatVector(np.reshape(data_X,(data_X.shape[0]*data_X.shape[1])))
    rdata_X=robjects.r.matrix(a,nrow=n,byrow='TRUE')

    b=robjects.FloatVector(np.reshape(data_Y,(data_Y.shape[0]*data_Y.shape[1])))
    rdata_Y=robjects.r.matrix(b,nrow=n,byrow='TRUE')
    
    atau=robjects.r.pcov(rdata_X,rdata_Y)
    
    datay_copy=np.copy(data_Y)
   
    p1_value=[]
    for i in range(100):
        np.random.shuffle(datay_copy)
        c=robjects.FloatVector(np.reshape(datay_copy,(data_Y.shape[0]*data_Y.shape[1])))
        rdatay_copy=robjects.r.matrix(c,nrow=n,byrow='TRUE')        
        
        phsic_value=robjects.r.pcov(rdata_X,rdatay_copy)
        p1_value.append(phsic_value)
            
            
    return np.mean(atau<np.array(p1_value))




import dcor
def com_DC(data_X,data_Y):
    atau=dcor.distance_correlation(data_Y,data_X)
    datay_copy=np.copy(data_Y)

    p1_value=[]
    for i in range(100):
        np.random.shuffle(datay_copy)
        
        p1_value.append(dcor.distance_correlation(datay_copy,data_X))

    return atau, p1_value

def test_DC(data_X,data_Y):
    atau=dcor.independence.distance_covariance_test(data_X, data_Y,num_resamples=200)
    p_value=list(atau)[0]
    return p_value



def com_significance(atau,atau_bt,alpha):
    if atau>np.percentile(atau_bt,100-alpha):
        return 1
    else:
        return 0
    

    
  

import rpy2.robjects as robjects
robjects.r("library(dcov)")

R_other=\
'''

###packages
library(clue)
library(adagio)
library(energy)
library(pracma)
library(randtoolbox)
library(RcppArmadillo)


library(mvtnorm)
library(purrr)
library(parallel)
require(randtoolbox, quietly = T)




###################  Shi et al. (2022) -Hallin- provided in tha supplementary material of Shi et al. (2022) ######
#####https://www.tandfonline.com/doi/full/10.1080/01621459.2020.1782223#supplemental-material-section
do <- function(p) { x = rnorm(p); return(x/sqrt(sum(x^2))) }

do.grid <- function(n,p){
  if (p == 1) {
    nR = n %/% 2; n0 = n %% 2
    g = matrix(c(-(nR:1)/(nR+1), rep(0,n0), (1:nR)/(nR+1)), n)
  } else {
    nR = floor(n^0.25); nS = n %/% nR; n0 = n %% nR
    g = rbind(((1:nR)/(nR+1)) %x% t(replicate(nS, do(p))), matrix(0,n0,p))
  }
  return(g)
}

Hallin.rank <- function(X, const=1e4, constC=1e8){
  n = nrow(X); p = ncol(X)
  X.Hallin = do.grid(n,p)
  if (p == 1){
    X.rank = X.Hallin[rank(X),]
  } else {
    X.DIST = matrix(NA,n,n)
    for (u in 1:n){
      for (v in 1:n){
        X.DIST[u,v] = sum((X[u,] - X.Hallin[v,])^2)
      }
    }
    if (const*max(X.DIST) < constC) {
      X.rank = X.Hallin[assignment(round(const*X.DIST))$perm,]
    } else {
      X.rank = X.Hallin[as.vector(solve_LSAP(X.DIST)),]
    }
  }
  return(X.rank)
}

discov.Hallin.stat <- function(X, Y, const=1e4, constC=1e8) {
  if (nrow(X) != nrow(Y)) {
    print("Error: sample sizes must agree!")
  } else {
    n = nrow(X)
    X.rank = Hallin.rank(X, const, constC)
    Y.rank = Hallin.rank(Y, const, constC)
    return(n*dcovU(X.rank, Y.rank))
  }
}


############# Deb and Sen (2023) -mrdCov- provided in the supplementary material of Deb and Sen (2023) #####
############ https://github.com/NabarunD
######### Computing Rank Energy Statistic #########
computestatisticrdcov=function(x,y,dim1=ncol(x),dim2=ncol(y),n=nrow(x),gridch=halton(n,dim1+dim2),gridch1=as.matrix(gridch[,(1:dim1)]),gridch2=as.matrix(gridch[,((dim1+1):(dim1+dim2))]))
{
  distmat1=matrix(0,nrow=n,ncol=n)
  for(i in 1:n)
    distmat1[i,]=apply((x[i,]-t(gridch1)),2,Norm)^2
  assignmentFUN1=solve_LSAP(distmat1)
  assignmentSOL1=cbind(seq_along(assignmentFUN1),assignmentFUN1)
  distmat2=matrix(0,nrow=n,ncol=n)
  for(i in 1:n)
    distmat2[i,]=apply((y[i,]-t(gridch2)),2,Norm)^2
  assignmentFUN2=solve_LSAP(distmat2)
  assignmentSOL2=cbind(seq_along(assignmentFUN2),assignmentFUN2)
  randcovSTAT=dcov.test(gridch1[assignmentSOL1[,2],],gridch2[assignmentSOL2[,2],], R=1)
  return(randcovSTAT$statistic)
}



        
'''




R_other2=\
'''

#################### Bcov ########################
library(Ball)
bcov_fun <- function(x,y,BstrpTimes=300){
   Bstrp <- matrix(0,BstrpTimes,1)
   Indep_Index =bcov(x,y);
   n=nrow(x)
   for (jj in 1:BstrpTimes){
     O=order(runif(n,0,1));
     YBstrp = y[O,];
     Bstrp[jj]  =bcov(x,YBstrp);
   }
   pVal = mean(Bstrp>=Indep_Index)
   return(pVal)  
  }


#################### mrdcov ########################

mrdcov_fun <- function(x,y,BstrpTimes=300){
   Bs_han <- matrix(0,BstrpTimes,1)
   n=nrow(x)
   had.inv <- try({ 
     Mrcov <- computestatisticrdcov(x,y)
     for (j in 1:BstrpTimes){
       O1=order(runif(n,0,1))
       YBstrp1 = y[O1,]
       Bs_han[j]  = computestatisticrdcov(x,YBstrp1)
     } 
   } , silent=TRUE)
   
   
   if ('try-error' %in% class(had.inv)) {
           Mrval <- 8888  # indicates failure; downstream code must check and skip
        }else{ 
      Mrval <- mean(Bs_han >= Mrcov)
     }
     
   return(Mrval)  
  }

            
    
####################### hallin #########################

hallin_fun <- function(x,y,BstrpTimes=300){
   Bs_han <- matrix(0,BstrpTimes,1)
   n=nrow(x)
   had.inv <- try({ 
     H_dcovt <- discov.Hallin.stat(x,y)
     for (j in 1:BstrpTimes){
       O1=order(runif(n,0,1))
       YBstrp1 = y[O1,]
       Bs_han[j]  = discov.Hallin.stat(x,YBstrp1)
     } 
   } , silent=TRUE)
   
   
   if ('try-error' %in% class(had.inv)) {
           HaVal <- 8888  # indicates failure; downstream code must check and skip
        }else{ 
      HaVal <- mean(Bs_han >= H_dcovt)
     }
     
   return(HaVal)  
  }

   
      
  
        
'''





R_other3='''
library(RcppArmadillo)

#Rcpp::sourceCpp('cv_hat_max_arma.cpp')

Rcpp::sourceCpp('/work/home/cas_tzt/cv_hat_max_arma.cpp')

#----------------------------------------------------- GCR -------------------------------------------------------------------------

######################    Coordinatewise   Gaussianization  ###############
#   - This is a column-wise, marginal Gaussianization.
data_transform <- function(x){
  n <- nrow(x)                            # sample size
  p <- ncol(x)                            # number of variables
  x_t2 <- matrix(0, nrow = n, ncol = p)   # generate n-by-p matrix
  
  for (j in 1:p) {                   # process each column independently
    # Calculate the empirical distribution function, scaled by n/(n+1) 
    ecdf_values <- ecdf(x[, j])(x[, j]) * n / (n + 1)
    x_t2[, j] <- ecdf_values
    
    for (i in 1:n) {                      
      if (x_t2[i, j] != 0 && x_t2[i, j] != 1) {
        x_t2[i, j] <- qnorm(x_t2[i, j])  # standard normal quantile at x_t2[i, j]
      } else {
        x_t2[i, j] <- 0                  # avoid Inf values 
      }
    }
  }
  
  list(x_trs = x, x_trs_my = x_t2)    
}



################# generate Mammen's multiplier: construct = "two-point mass" ################
rmammen <- function(n,
                    construct = c("normal-2", "normal-1", "two-point mass")){
  construct <- match.arg(construct)
  
  if (length(n) > 1) n <- length(n)
  
  if (construct == "two-point mass"){
    .vals <- c(-(sqrt(5)-1)/2, (sqrt(5)+1)/2) 
    .probs <- rev(abs(.vals)/sqrt(5))
    
    sample(.vals, size = n, replace = TRUE, prob = .probs)    #generate random samples taking values ±(sqrt(5)±1)/2 with probabilities ensuring mean 0 and variance 1.
  } else if (construct == "normal-1"){
    .v <- rnorm(n)
    
    (.v/sqrt(2)) + (0.5*((.v*.v) - 1))
  } else {
    .delta1 <- sqrt((3/4) + (sqrt(17)/12))
    .delta2 <- sqrt((3/4) - (sqrt(17)/12))
    .v <- matrix(rnorm(2*n), nrow=n, ncol=2)
    
    (.delta1 + (.v[,1]/sqrt(2)))*(.delta2 + (.v[,2]/sqrt(2))) -
      (.delta1*.delta2)
  }
}




# The proposed independence test with multiplier bootstrap
# Input:
#   x, y:   data matrices with the same number of rows (sample size n)
#   alpha:  significance level in (0, 1)
#   seed:   RNG seed (optional)
#   option: "Rademacher" | "Gaussian" | "Mammen" | "all"
#     "Rademacher" uses Rademacher multipliers;
#     "Gaussian" uses Gaussian multipliers;
#     "Mammen" uses Mammen's multipliers;
#     "all" computes results for all three types of multipliers
#   N: number of bootstrap replications, default 5000

# Output:
#   option:   the chosen option
#   reject:   1: reject the null hypothesis; 0: do not reject the null hypothesis
#   p_value:  p value
#   test_sta: test statistic
#   cv:       critical value

Ind_Gtest <- function(x, y, alpha, seed = 1,
                      option = c("Rademacher", "Gaussian", "Mammen", "all"),
                      N = 5000) {
  option <- match.arg(option)
  
  # ---- ensure x has the larger column dimension ----
  if (ncol(x) < ncol(y)) {
    tmp <- x; x <- y; y <- tmp
  }
  
  # ---- coordinatewise Gaussianization ----
  x_norm <- data_transform(x)$x_trs_my
  y_norm <- data_transform(y)$x_trs_my
  
  # ---- Generate multiplier matrices: N-by-nrow(x) ----
  make_M <- function(kind) {
    B <- N * nrow(x)
    if (kind == "Rademacher") {
      set.seed(seed)
      Mvec <- sample(c(-1, 1), size = B, replace = TRUE, prob = c(0.5, 0.5))  # generate Rademacher multipliers
    } else if (kind == "Gaussian") {
      set.seed(seed)
      Mvec <- rnorm(B, mean = 0, sd = 1)          # generate Gaussian multipliers
    } else if (kind == "Mammen") {
      set.seed(seed)
      Mvec <- rmammen(B, "two-point mass")        # generate Mammen's multipliers
    } else {
      stop("Unknown kind: ", kind)
    }
    matrix(Mvec, nrow = N)       # an N-by-n matrix
  }
  
  # ---- call function cv_arma (cv_hat_max_arma.cpp) for a given multiplier ----
  run_once <- function(kind) {
    M   <- make_M(kind)                  # generate multiplier
    res <- cv_arma(x_norm, y_norm, M, alpha)   
    tsta <- res$ts                       # test statistic
    cv   <- res$cv_e                     # critical value
    reject <- as.numeric(tsta > cv)      # reject the null hypothesis (=1) if the test statistic > critical value
    list(
      option   = kind,
      reject   = reject,          
      p_value  = res$p_value,            # p value 
      test_sta = tsta,            
      cv       = cv               
    )
  }
  
  if (option == "all") {
    out_r <- run_once("Rademacher")
    out_g <- run_once("Gaussian")
    out_m <- run_once("Mammen")
    return(list(
      Rademacher = out_r,
      Gaussian   = out_g,
      Mammen     = out_m
    ))
  } else {
    return(run_once(option))
  }
}
'''


#gauss-------------------------------------------------------------------------
R_gauss='''
######################    Coordinatewise   Gaussianization  ###############
#   - This is a column-wise, marginal Gaussianization.
data_transform <- function(x){
  n <- nrow(x)                            # sample size
  p <- ncol(x)                            # number of variables
  x_t2 <- matrix(0, nrow = n, ncol = p)   # generate n-by-p matrix
  
  for (j in 1:p) {                   # process each column independently
    # Calculate the empirical distribution function, scaled by n/(n+1) 
    ecdf_values <- ecdf(x[, j])(x[, j]) * n / (n + 1)
    x_t2[, j] <- ecdf_values
    
    for (i in 1:n) {                      
      if (x_t2[i, j] != 0 && x_t2[i, j] != 1) {
        x_t2[i, j] <- qnorm(x_t2[i, j])  # standard normal quantile at x_t2[i, j]
      } else {
        x_t2[i, j] <- 0                  # avoid Inf values 
      }
    }
  }
  
  list(x_trs = x, x_trs_my = x_t2)    
}
'''

def data_transform(rdata):
    rdata=rdata_transform(rdata)
    data_Xg=np.array(rdata)[1]
    return data_Xg



from rpy2.robjects import globalenv


robjects.r(R_other)
robjects.r(R_other2)

robjects.r(R_gauss)
rdata_transform=globalenv['data_transform']

robjects.r(R_other3)


mrdcov=globalenv['mrdcov_fun']
Hallin=globalenv['hallin_fun']

BCov=globalenv['bcov_fun']


from hyppo.d_variate import dHsic
from hyppo.independence import HHG
CGR=globalenv['Ind_Gtest']

